In [ ]:
#| default_exp meta_learning.environments.pricing_env.pricing_env

In [ ]:
#| export
import gym
from abc import ABC, abstractmethod
from typing import Union, List, Dict, Optional
import numpy as np


In [ ]:
#| export

class PricingEnv(gym.Env):
    """
    ──────────────────────────────────────────────────────────────────────────
    Single-SKU pricing environment for meta-RL (RL², variBAD, etc.)
    ──────────────────────────────────────────────────────────────────────────

    Task vector  (length = 2 · F + 3)  
        [ alpha ₀,…,alpha _{F-1},
          beta ₀,…,beta _{F-1},
          sigma,
          inv_ratio,
          horizon ]

      • *alpha* (F)    – intercept terms of the linear demand model  
      • *beta*  (F)    – slope terms (negative ⇒ demand ↓ as price ↑)  
      • *sigma*        – st.dev. of additive Gaussian demand noise  
      • *inv_ratio*    – initial inventory / horizon (0 … 1)  
      • *horizon*      – episode length (integer)

    Observation  
        [ inventory , feature₀ … feature_{F-1} ]

    Action  
        scalar price in ⁠[ p_low , p_high ].

    Reward  
        revenue = price × sales  (no holding- or stock-out costs).

    Notes for meta-RL wrappers
    --------------------------
    • **`reset_task()`** changes *only* the latent task; it does *not* start
      an episode. Call `reset()` afterwards.  
    • `info["task"]` returns the current task vector for logging /
      evaluation.  
    • Boundaries of `task_space` are set to large finite values only to keep
      Gym quiet; the code never samples from that space.

    Reproducibility
    ---------------
    This minimal version uses the **global NumPy RNG** (`np.random`).  If you
    need deterministic roll-outs in a vectorised setup, create your own
    wrapper that seeds the RNG per worker.
    """

    _BIG = 1e9  # very wide bound for observation & task spaces

    # --------------------------------------------------------------------- #
    #                         constructor                                    #
    # --------------------------------------------------------------------- #
    def __init__(self,
                 # price limits
                 p_low: float = 0.0,
                 p_high: float = 1.0,

                 # feature & task-distribution settings
                 nb_features: int = 1,
                 horizon_choices: List[int] = (500,),
                 mean_alpha: List[float] = (1.2,),
                 std_alpha: float = 0.0,
                 mean_beta: List[float] = (-0.3,),
                 std_beta: float = 0.0,
                 noise_std_choices: List[float] = (0.2,),
                 inv_ratio_mean: float = 0.5,
                 inv_ratio_std: float = 0.1,
                 function_choices: List[int] = (0,), # 0: linear, 1: log

                 # optional fixed task (otherwise sampled)
                 task: Optional[np.ndarray] = None):

        super().__init__()

        # -------- store hyper-parameters -------------------------------------
        self.nb_features       = nb_features
        self.horizon_choices   = tuple(horizon_choices)
        self.mean_alpha        = tuple(mean_alpha)
        self.std_alpha         = std_alpha
        self.mean_beta         = tuple(mean_beta)
        self.std_beta          = std_beta
        self.noise_std_choices = tuple(noise_std_choices)
        self.inv_ratio_mean    = inv_ratio_mean
        self.inv_ratio_std     = inv_ratio_std
        self.function_choices     = tuple(function_choices)
        # -------- Gym spaces --------------------------------------------------
        # action: scalar price -------------------------------------------------
        self.action_space = gym.spaces.Box(
            low=np.array([p_low],  dtype=np.float32),
            high=np.array([p_high], dtype=np.float32),
            dtype=np.float32
        )

        # observation: inventory + features -----------------------------------
        obs_dim = 1 + nb_features
        self.observation_space = gym.spaces.Box(
            low=-self._BIG, high=self._BIG,
            shape=(obs_dim,), dtype=np.float32
        )

        # task vector (never actually sampled by the library code) ------------
        self.task_dim = 2 * nb_features + 4
        self.task_space = gym.spaces.Box(
            low=-self._BIG, high=self._BIG,
            shape=(self.task_dim,), dtype=np.float32
        )
        self.seed()
        # -------- set latent task & episode ----------------------------------
        self.reset_task(task)
        self.reset()

    # ===================================================================== #
    #                      meta-learning hooks                               #
    # ===================================================================== #
    def reset_task(self, task: Optional[np.ndarray] = None) -> None:
        """
        Pick a new latent task.

        Parameters
        ----------
        task : ndarray or None
            • *None*  – sample from the task prior  
            • ndarray – use given vector (length == `task_dim`)
        """
        if task is None:
            task = self._sample_task()
        task = np.asarray(task, dtype=np.float32)
        assert task.shape == (self.task_dim,), f"task has wrong shape {task.shape}"
        self._task = task

        F = self.nb_features
        self.alpha     = task[:F]
        self.beta      = task[F:2*F]
        self.sigma     = float(task[2*F])
        self.inv_ratio = float(np.clip(task[2*F + 1], 0.00, 1.0))
        self.horizon   = int(task[2*F + 2])
        self.function = int(task[2*F + 3])

    def get_task(self) -> np.ndarray:
        """Return **copy** of the current task vector (float32)."""
        return self._task.copy()

    # ===================================================================== #
    #                       Gym interface                                   #
    # ===================================================================== #
    def reset(self):
        """Start a fresh episode under the *current* task."""
        self.t   = 0
        self.inv = float(self.horizon * self.inv_ratio)
        return self._get_obs()

    def step(self, action):
        done = False
        # ── 1. clip price into valid range ---------------------------------------
        price = float(np.clip(action, self.action_space.low[0],
                                    self.action_space.high[0]))

        # ── 2. realise (stochastic) demand ---------------------------------------
        noise   = np.random.normal(0.0, self.sigma)
        demand  = self._demand(price, noise)

        # ── 3. apply / skip inventory constraint ---------------------------------
        if self.inv_ratio == 0.0:                # ▶ unlimited stock
            sales = demand                       #   no inventory depletion
        else:                                    # ▶ finite stock
            sales = min(demand, self.inv)
            self.inv -= sales
            done = (self.inv <= 0.0)  # stock-out if inv drops to zero

        # ── 4. reward (revenue) ---------------------------------------------------
        reward = price * sales

        # ── 5. book-keeping -------------------------------------------------------
        self.t += 1
        done = done or (self.t >= self.horizon)

        obs = self._get_obs()
        info = {
            "task":   self.get_task(),
            "noise":  noise,
            "demand": demand,
            "sales":  sales,
            "inv":    self.inv
        }
        return obs, reward, done, info

    def seed(self, seed=None):
        self.np_random, seed = gym.utils.seeding.np_random(seed)
        return [seed]
    # ===================================================================== #
    #                         internal helpers                               #
    # ===================================================================== #
    # ---------- task sampler --------------------------------------------------
    def _sample_task(self) -> np.ndarray:
        F = self.nb_features
        mean_alpha = float(np.random.choice(self.mean_alpha))
        mean_beta  = float(np.random.choice(self.mean_beta))
        alpha = np.random.normal(mean_alpha, self.std_alpha,  size=F)
        beta  = np.random.normal(mean_beta,  self.std_beta,   size=F)
        sigma = float(np.random.choice(self.noise_std_choices))
        inv_ratio = float(np.clip(
            np.random.normal(self.inv_ratio_mean, self.inv_ratio_std), 0.00, 1.0))
        horizon = int(np.random.choice(self.horizon_choices))
        function_choices = np.random.choice(self.function_choices)
        return np.concatenate([alpha, beta,
                               [sigma, inv_ratio, horizon, function_choices]]).astype(np.float32)

    # ---------- observation helpers ------------------------------------------
    def _get_features(self) -> np.ndarray:
        """Draw a single feature vector Xₜ."""
        if self.nb_features == 1:
            return np.ones(1, dtype=np.float32)
        scale = 1.0 / np.sqrt(self.nb_features - 1)
        return np.random.uniform(0.0, scale, size=self.nb_features).astype(np.float32)

    def _get_obs(self) -> np.ndarray:
        self._X = self._get_features()
        return np.concatenate([self._X, [self.inv]]).astype(np.float32)

    # ---------- demand model --------------------------------------------------
    def _demand(self, price: float, noise: float) -> float:
        """Linear demand with additive noise; demand ≥ 0."""
        if self.function == 1: # exp function
            mean = float(np.exp(np.dot(self._X, self.alpha)* 0.55 + np.dot(self._X, self.beta) * price))
        
        elif self.function == 2: # logit function
            zeta = np.exp(np.dot(self._X, self.alpha) + np.dot(self._X, self.beta) * price)
            mean = float(3 * np.divide(zeta, 1 + zeta)) 
            
        else: # Linear function
            mean = float(np.dot(self._X, self.alpha) + np.dot(self._X, self.beta) * price)
        return max(0.0, mean + noise)

    # ---------- visualisation stub -------------------------------------------
    def visualise_behaviour(self, *_, **__):
        """
        Optional — leave blank.  Hyper’s default visualiser is used if None.
        """
        return None, None, None, None, None, None, None


In [ ]:
# Create a test environment
test_env = PricingEnv(p_low=0, p_high=5, 
                      nb_features=1, horizon_choices=[500],
                      mean_alpha=[1.2], std_alpha=0.0,
                      mean_beta=[-0.3], std_beta=0.0,
                      noise_std_choices=[0.2], 
                      inv_ratio_mean=0.0, 
                      inv_ratio_std=0.0,
                      function_choices=[0, 1, 2])  # linear demand

In [ ]:
# Reset the task
test_env.reset_task()
test_env.get_task()

array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  2.0e+00],
      dtype=float32)

array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  0.0e+00],
      dtype=float32)

In [ ]:
test_env.function

0

In [ ]:
test_env.reset()

array([1., 0.], dtype=float32)

In [ ]:
action = np.array([2])  # Example action within the bounds
test_env.step(action)

(array([1., 0.], dtype=float32),
 1.150863826801591,
 False,
 {'task': array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  0.0e+00],
        dtype=float32),
  'noise': -0.024568110441062385,
  'demand': 0.5754319134007955,
  'sales': 0.5754319134007955,
  'inv': 0.0})

In [ ]:
test_env.reset()
done = False
step_counter = 0  # Initialize the counter
while not done:
    action = np.array([2])  # Random action within bounds
    obs, reward, done, info = test_env.step(action)
    step_counter += 1  # Increment the counter
    print(f"Step: {step_counter}, Obs: {obs}, Reward: {reward}, Done: {done}, Info: {info}")

Step: 1, Obs: [1. 0.], Reward: 1.6445322575257988, Done: False, Info: {'task': array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  0.0e+00],
      dtype=float32), 'noise': 0.22226610492104143, 'demand': 0.8222661287628994, 'sales': 0.8222661287628994, 'inv': 0.0}
Step: 2, Obs: [1. 0.], Reward: 1.080863778704961, Done: False, Info: {'task': array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  0.0e+00],
      dtype=float32), 'noise': -0.05956813448937735, 'demand': 0.5404318893524805, 'sales': 0.5404318893524805, 'inv': 0.0}
Step: 3, Obs: [1. 0.], Reward: 1.0086724437559227, Done: False, Info: {'task': array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  0.0e+00],
      dtype=float32), 'noise': -0.09566380196389654, 'demand': 0.5043362218779613, 'sales': 0.5043362218779613, 'inv': 0.0}
Step: 4, Obs: [1. 0.], Reward: 0.9065110947355939, Done: False, Info: {'task': array([ 1.2e+00, -3.0e-01,  2.0e-01,  0.0e+00,  5.0e+02,  0.0e+00],
      dtype=float32), 'noise': -0.14674447